In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
print(ROOT)

c:\Users\HP\Desktop\Ejust\Projects\Data Security\nids


In [5]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix
from src.pipeline import DATA, RESULTS, load_scaler, iter_split

SEED = 11
cols, scaler = load_scaler()
print(ROOT, len(cols), "features")

c:\Users\HP\Desktop\Ejust\Projects\Data Security\nids 40 features


In [6]:
from src.pipeline import iter_split

def get(name, cap=30_000, n_benign=100_000):
    df = pd.concat(list(iter_split(name, cols, cap=cap)))
    b = df[df["Attack"] == "Benign"]
    b = b.sample(min(n_benign, len(b)), random_state=SEED)
    return pd.concat([b, df[df["Attack"] != "Benign"]])

tr = pd.read_parquet(DATA / "train.parquet", columns=cols + ["Attack"]).sample(200_000, random_state=SEED)
va, tu, te = get("val"), get("tune"), get("test")

groups = {
    "train_benign": tr,
    "val_benign": va,
    "tune_benign": tu[tu["Attack"] == "Benign"],
    "tune_attack": tu[tu["Attack"] != "Benign"],
    "test_benign": te[te["Attack"] == "Benign"],
    "test_attack": te[te["Attack"] != "Benign"],
}
feats = [c for c in ["DURATION_OUT", "DURATION_IN", "RETRANSMITTED_IN_PKTS",
                     "RETRANSMITTED_IN_BYTES", "IN_BYTES", "LONGEST_FLOW_PKT"] if c in cols]

print("fraction of flows with value > 0")
print(pd.DataFrame({g: (d[feats] > 0).mean() for g, d in groups.items()}).round(3))
print("\n99th percentile (raw values)")
print(pd.DataFrame({g: d[feats].quantile(0.99) for g, d in groups.items()}).round(1))
print("\ntrain std after log1p (what the scaler divides by)")
print(pd.Series(scaler.scale_, index=cols)[feats].round(4))

fraction of flows with value > 0
                        train_benign  val_benign  tune_benign  tune_attack  \
DURATION_OUT                   0.002       0.003        0.004        0.010   
DURATION_IN                    0.008       0.017        0.011        0.018   
RETRANSMITTED_IN_PKTS          0.000       0.000        0.001        0.001   
RETRANSMITTED_IN_BYTES         0.000       0.000        0.001        0.001   
IN_BYTES                       1.000       1.000        1.000        1.000   
LONGEST_FLOW_PKT               1.000       1.000        1.000        1.000   

                        test_benign  test_attack  
DURATION_OUT                  0.265        0.223  
DURATION_IN                   0.334        0.223  
RETRANSMITTED_IN_PKTS         0.047        0.026  
RETRANSMITTED_IN_BYTES        0.047        0.026  
IN_BYTES                      1.000        1.000  
LONGEST_FLOW_PKT              1.000        1.000  

99th percentile (raw values)
                        train_ben